# Proyecto 2 — Clasificación de Décadas Históricas en Español
## Aprendizaje de Máquina 2026-10

Alicia Robinson — 202321278 | N Felipe Celis D — 202320636
Isabella Naranjo — 202321096 | David Caro — 202222073

---

## Resumen del proceso de iteración

Este notebook es el resultado de múltiples iteraciones. A continuación describimos cómo llegamos a la solución final y qué aprendimos en el camino.

**Iteración 1 — Modelo básico BERT con freeze schedule (F1 = 0.23)**
El primer intento usó BETO con un esquema de "descongelado gradual" (freeze schedule): en la primera época solo entrenaba la cabeza de clasificación, y en épocas posteriores se iban liberando capas. El resultado fue muy bajo porque solo había una época con todos los parámetros activos, lo que no es suficiente para que el encoder se adapte al problema. Adicionalmente, usábamos mean pooling del encoder cuando BETO fue preentrenado para usar el token [CLS]. Ambos errores combinados producían F1 ≈ 0.23 en validación, que es peor que el TF-IDF solo.

**Iteración 2 — TF-IDF recuperado de la Parte 1 (F1 ≈ 0.30)**
Al observar que el transformer no aportaba nada, volvimos a la receta probada de la Parte 1: 4 vectorizadores TF-IDF ponderados con GlobalSpecialistClassifier. Este modelo solo da F1 ≈ 0.30 en Kaggle y sirvió como punto de partida y red de seguridad.

**Iteración 3 — BERTIN (F1 OOF = 0.2674)**
Probamos el modelo BERTIN (RoBERTa entrenado en español) con features estilométricas concatenadas al [CLS]. Mejoró ligeramente respecto a la iteración 1 pero seguía siendo inferior al TF-IDF en validación. El ensemble dio 0.32282 en Kaggle.

**Iteración 4 — RoBERTa-BNE + ensemble inteligente (versión actual)**
Cambiamos a RoBERTa-base-BNE (Biblioteca Nacional de España), el único modelo preentrenado en corpus histórico español. Agregamos una política de ensemble inteligente: si el transformer no supera al TF-IDF en validación OOF, el submission final es TF-IDF puro. Esto garantiza que el transformer solo aporta si realmente mejora.

## 0. Compatibilidad de PyTorch con la GPU de Kaggle

PyTorch 2.4 y versiones posteriores eliminaron el soporte para la GPU P100 de Kaggle (arquitectura sm_60). Esto causaba el error "no kernel image is available for execution on the device" que bloqueaba completamente el entrenamiento.

La solución es instalar PyTorch 2.3.1 que sí es compatible con P100, T4 y A100. Esta celda verifica la versión instalada y reinstala solo si es necesario.

Este fix fue identificado después de varios intentos fallidos donde el training crasheaba silenciosamente. Es una de las correcciones que diferencian esta versión de las anteriores.

In [ ]:
# PyTorch 2.4+ eliminó soporte para P100 (sm_60).
# 2.3.1 funciona en P100, T4 y A100 sin problemas.
import subprocess, sys

def get_torch_version():
    try:
        import torch; return tuple(int(x) for x in torch.__version__.split('.')[:2])
    except: return (0, 0)

major, minor = get_torch_version()
print(f'PyTorch actual: {major}.{minor}')

if major > 2 or (major == 2 and minor >= 4):
    print('Versión demasiado nueva para P100 — reinstalando 2.3.1...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.3.1', 'torchvision==0.18.1',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ], check=True)
    print('Reinstalación OK — REINICIA EL KERNEL si hay problemas')
else:
    print('Versión OK, no se necesita reinstalar')


PyTorch actual: 2.3
Versión OK, no se necesita reinstalar


## 1. Imports y configuración

Importamos las librerías necesarias y fijamos la semilla aleatoria (SEED=42) para reproducibilidad completa. Usamos PyTorch como framework de deep learning y HuggingFace Transformers para el modelo preentrenado. Todo el código está diseñado para correr en GPU (necesario para el transformer) pero el TF-IDF corre en CPU.

La variable DEVICE detecta automáticamente si hay GPU disponible. En Kaggle se debe activar GPU T4 desde Session Options antes de ejecutar.

In [ ]:
import os, re, json, gc, warnings, random, time
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
from scipy.special import softmax
from scipy.optimize import minimize

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Parche torch.load para PyTorch 2.6+
_orig_load = torch.load
def _patched_load(f, *a, **kw):
    kw.setdefault('weights_only', False)
    return _orig_load(f, *a, **kw)
torch.load = _patched_load

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)

# Buscar CSVs en /kaggle/input o localmente
import glob
def find_csv(name):
    m = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return m[0] if m else name

TRAIN_PATH = find_csv('train.csv')
EVAL_PATH  = find_csv('eval.csv')

print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    # Test rápido de CUDA
    try:
        _x = torch.zeros(2,2).cuda() + 1
        print(f'CUDA OK : {_x[0,0].item():.0f}')
        del _x
    except Exception as e:
        print(f'CUDA ERROR: {e}')
        print('Si ves este error, reinstala PyTorch con la celda anterior y reinicia el kernel')
print(f'Train   : {TRAIN_PATH}')
print(f'Eval    : {EVAL_PATH}')


Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB
CUDA OK : 1
Train   : /kaggle/input/datasets/alicia3007/parte2/train.csv
Eval    : /kaggle/input/datasets/alicia3007/parte2/eval.csv


## 2. Carga, EDA y preprocesamiento

**Decisión clave: limpieza mínima.**

En la Entrega 1 descubrimos que limpiar demasiado el texto destruye señales temporales importantes. Por ejemplo, la letra "ſ" (s larga) aparece en textos anteriores a 1810 y es un indicador temporal muy fuerte. Si la normalizamos a "s", perdemos esa señal.

Lo que sí limpiamos:
- Palabras partidas por guión de fin de línea ("pala-\nbra" → "palabra"): esto es ruido tipográfico del OCR, no señal histórica.
- Boilerplate de Google Books: frases genéricas de copyright que no pertenecen al texto original.
- Duplicados exactos y label noise: textos idénticos con décadas distintas asignadas, que son errores del dataset.

Lo que NO tocamos:
- Mayúsculas (cambian entre siglos)
- Acentos (señal ortográfica histórica)
- Caracteres especiales históricos
- Grafías arcaicas (u/v, i/j, quando/cual)

In [ ]:
_BP = re.compile('|'.join([
    'google books','google book search',r'books\.google\.com',
    'digitized by google','digitalizado por google',
    'scanned by google','keep it legal','maintain attribution',
    'non-commercial use','copyright infringement','this is a digital copy',
    'usage guidelines','watermark',
]), re.IGNORECASE)

def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'(\w+)-\s*\n\s*', r'\1', text)
    text = re.sub(r'(\w+)\u00ac\s*\n\s*', r'\1', text)
    text = text.replace('\n',' ').replace('\r',' ')
    text = re.sub(r'[ \t]{2,}',' ',text)
    return text.strip()

def load_csv(path):
    for enc in ('utf-8','latin-1'):
        try: return pd.read_csv(path, engine='python', on_bad_lines='skip', encoding=enc)
        except: continue
    return pd.read_csv(path)

train_raw = load_csv(TRAIN_PATH)
eval_raw  = load_csv(EVAL_PATH)

train = train_raw.copy()
train = train.dropna(subset=['text','decade']).reset_index(drop=True)
train['text']   = train['text'].astype(str).apply(clean_text)
train['decade'] = pd.to_numeric(train['decade'], errors='coerce')
train = train.dropna(subset=['decade']).reset_index(drop=True)
train['decade'] = train['decade'].astype(int)
train = train[train['text'].str.len() > 10].copy()
train = train[~train['text'].str.contains(_BP, na=False)].copy()
train = train.drop_duplicates(subset=['text','decade']).copy()
dup = train.duplicated(subset=['text'], keep=False)
noise = set()
for txt, grp in train[dup].groupby('text'):
    if grp['decade'].nunique() > 1:
        noise.update(grp.index.tolist())
train = train[~train.index.isin(noise)].reset_index(drop=True)

eval_df = eval_raw.copy()
eval_df['text'] = eval_df['text'].astype(str).fillna('').apply(clean_text)
eval_df['id']   = pd.to_numeric(eval_df['id'], errors='coerce').astype('Int64')
eval_df = eval_df[eval_df['text'].str.len() > 0].reset_index(drop=True)

# Leakage map
tn = train['text'].str.strip().str.lower()
en = eval_df['text'].str.strip().str.lower()
LEAKAGE_MAP = {}
for _, row in eval_df[en.isin(tn)].iterrows():
    m = train[tn == row['text'].strip().lower()]
    if len(m): LEAKAGE_MAP[int(row['id'])] = int(m['decade'].mode()[0])

# GroupKFold doc_id — CLAVE de v82: previene leakage de fragmentos del mismo doc
train['source_doc_id'] = (train['text'].str[:50].apply(hash) % 10_000).abs()

labels_global = sorted(train['decade'].unique().tolist())
label2id = {d: i for i,d in enumerate(labels_global)}
id2label = {i: d for i,d in enumerate(labels_global)}
num_labels = len(labels_global)
train['label'] = train['decade'].map(label2id)

X_full = train['text'].values
y_full = train['decade'].astype(int).values

print(f'Train : {len(train):,} | Eval : {len(eval_df):,}')
print(f'Clases: {num_labels} ({labels_global[0]}–{labels_global[-1]})')
print(f'Leakage: {len(LEAKAGE_MAP)} IDs')
print(f'GroupKFold source_doc_id: {train["source_doc_id"].nunique()} docs únicos')


Train : 31,321 | Eval : 3,490
Clases: 39 (150–188)
Leakage: 9 IDs
GroupKFold source_doc_id: 9565 docs únicos


## 3. TF-IDF con GlobalSpecialistClassifier

Este es el la parte principal del sistema y viene directamente de la Entrega 1, donde fue uno de los modelos más fuertes de la competencia.

Conservamos la arquitectura de dos niveles sin cambios porque demostró empíricamente ser superior a alternativas más complejas:

El modelo global usa cuatro representaciones TF-IDF en paralelo con pesos diferenciados. Los vectorizadores de caracteres pesan más que el de palabras porque capturan patrones ortográficos aunque la grafía varíe entre siglos. El vectorizador de palabras preserva los acentos (strip_accents=None) porque son parte de la señal histórica, decisión que también viene de la Parte 1. El VotingClassifier combina Regresión Logística, SVM calibrada y SGD — cada uno comete errores distintos y al promediar probabilidades el resultado es más robusto.

Los 8 especialistas por bloque de décadas fueron la innovación principal de la Parte 1. Cuando el global predice que un texto pertenece a cierto bloque temporal, el especialista de ese bloque refina la predicción dentro de las 5 décadas de su rango. Esto corrige errores entre décadas cercanas que comparten vocabulario. Los parámetros son idénticos a los del mejor modelo de la Parte 1: global_lr_C=5.0, global_svc_C=0.3, specialist_lr_C=3.0, specialist_svc_C=0.2.

El TF-IDF se guarda en disco inmediatamente al terminar. Si la sesión de Kaggle se cae, el modelo se recarga sin reentrenar. Esta estrategia de checkpoint continuo la aplicamos en todas las partes del pipeline.

In [ ]:
def build_feature_union(char_wb_short_max=120_000, char_wb_long_max=150_000,
                        char_nowb_max=100_000, word_max=170_000):
    return FeatureUnion([
        ('char_short', TfidfVectorizer(
            lowercase=False, analyzer='char_wb', ngram_range=(1,3),
            max_features=char_wb_short_max, min_df=2, max_df=0.95,
            sublinear_tf=True, strip_accents='unicode', dtype=np.float32)),
        ('char_long', TfidfVectorizer(
            lowercase=False, analyzer='char_wb', ngram_range=(3,6),
            max_features=char_wb_long_max, min_df=2, max_df=0.95,
            sublinear_tf=True, strip_accents='unicode', dtype=np.float32)),
        ('char_nowb', TfidfVectorizer(
            lowercase=False, analyzer='char', ngram_range=(2,6),
            max_features=char_nowb_max, min_df=2, max_df=0.95,
            sublinear_tf=True, strip_accents='unicode', dtype=np.float32)),
        ('word', TfidfVectorizer(
            lowercase=False, analyzer='word', ngram_range=(1,3),
            max_features=word_max, min_df=2, max_df=0.95,
            sublinear_tf=True, strip_accents=None, dtype=np.float32)),
    ], transformer_weights={'char_short':1.5,'char_long':1.5,'char_nowb':1.3,'word':1.0})

def build_pipeline(lr_C=5.0, svc_C=0.3, word_max=170_000):
    fu  = build_feature_union(word_max=word_max)
    lr  = LogisticRegression(C=lr_C, solver='saga', max_iter=350, tol=2e-3,
                              random_state=SEED, n_jobs=1)
    svc = CalibratedClassifierCV(LinearSVC(C=svc_C, max_iter=2000,
                                            random_state=SEED), cv=3, method='isotonic')
    sgd = SGDClassifier(loss='modified_huber', alpha=1e-4, max_iter=200,
                         random_state=SEED, n_jobs=1)
    return Pipeline([('f', fu),
                      ('c', VotingClassifier([('lr',lr),('sv',svc),('sg',sgd)],
                                              voting='soft'))])

class GlobalSpecialistClassifier(BaseEstimator, ClassifierMixin):
    """Modelo global + 8 especialistas por bloque de décadas."""
    BLOCKS = {
        'b150': list(range(150,155)), 'b155': list(range(155,160)),
        'b160': list(range(160,165)), 'b165': list(range(165,170)),
        'b170': list(range(170,175)), 'b175': list(range(175,180)),
        'b180': list(range(180,185)), 'b185': list(range(185,189)),
    }
    def __init__(self, global_lr_C=5.0, global_svc_C=0.3,
                 specialist_lr_C=3.0, specialist_svc_C=0.2,
                 word_max=170_000, verbose=True):
        self.global_lr_C=global_lr_C; self.global_svc_C=global_svc_C
        self.specialist_lr_C=specialist_lr_C; self.specialist_svc_C=specialist_svc_C
        self.word_max=word_max; self.verbose=verbose

    def fit(self, X, y):
        X = pd.Series(X).fillna('').astype(str).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        if self.verbose: print('  [Global] entrenando...')
        self.gm_ = build_pipeline(self.global_lr_C, self.global_svc_C, self.word_max)
        self.gm_.fit(X, y)
        self.sp_ = {}
        for bn, bc in self.BLOCKS.items():
            m = y.isin(bc)
            if m.sum() < 20: continue
            if self.verbose: print(f'  [{bn}] n={m.sum()} entrenando...')
            sp = build_pipeline(self.specialist_lr_C, self.specialist_svc_C, self.word_max)
            sp.fit(X[m], y[m])
            self.sp_[bn] = (sp, bc)
        return self

    def predict(self, X):
        X = pd.Series(X).fillna('').astype(str).reset_index(drop=True)
        fp = np.array(self.gm_.predict(X), dtype=int)
        for bn, (sp, bc) in self.sp_.items():
            idx = np.where(np.isin(fp, bc))[0]
            if len(idx): fp[idx] = sp.predict(X.iloc[idx]).astype(int)
        return fp

    def predict_proba(self, X):
        X = pd.Series(X).fillna('').astype(str).reset_index(drop=True)
        return self.gm_.predict_proba(X)

    @property
    def classes_(self): return self.gm_.classes_

print('GlobalSpecialistClassifier definido')


GlobalSpecialistClassifier definido


## 4. Entrenamiento TF-IDF y submission de respaldo

Una lección importante de sesiones anteriores de las iteraciones: nunca esperar al final para generar el primer submission. Si la sesión de Kaggle se cae antes de terminar el transformer (que puede tardar 3 horas), perderíamos todo.

Por eso generamos submission_tfidf.csv inmediatamente después de entrenar el TF-IDF. Este archivo ya es válido para subir a Kaggle y representa nuestro piso garantizado — el mismo nivel que logramos en la Parte 1.

In [ ]:
TFIDF_PATH = OUTPUT_DIR / 'model_tfidf_v84.joblib'

if TFIDF_PATH.exists():
    print(f'Cargando TF-IDF desde {TFIDF_PATH}...')
    model_tfidf = joblib.load(TFIDF_PATH)
else:
    print(f'Entrenando GlobalSpecialistClassifier (N={len(X_full):,})...')
    t0 = time.time()
    model_tfidf = GlobalSpecialistClassifier(verbose=True)
    model_tfidf.fit(X_full, y_full)
    joblib.dump(model_tfidf, TFIDF_PATH)
    print(f'Guardado en {TFIDF_PATH} ({TFIDF_PATH.stat().st_size/1e6:.1f} MB)')
    print(f'Tiempo: {(time.time()-t0)/60:.1f} min')

# Submission de respaldo inmediato
preds_tfidf = model_tfidf.predict(eval_df['text'].values)
sub_tfidf = pd.DataFrame({'id': eval_df['id'].astype(int).values,
                           'answer': preds_tfidf.astype(int)})
for eid, dec in LEAKAGE_MAP.items():
    m = sub_tfidf['id'] == eid
    if m.any(): sub_tfidf.loc[m, 'answer'] = int(dec)
assert sub_tfidf['answer'].between(150,188).all()
sub_tfidf.to_csv(OUTPUT_DIR / 'submission_tfidf.csv', index=False)
sub_tfidf.to_csv('submission_tfidf.csv', index=False)
print('>>> submission_tfidf.csv guardado (respaldo seguro) <<<')


Cargando TF-IDF desde outputs/model_tfidf_v84.joblib...
>>> submission_tfidf.csv guardado (respaldo seguro) <<<


## 4b. OOF con GroupKFold — el cambio más importante respecto a la Parte 1

En la Parte 1 usamos StratifiedKFold para validación cruzada. Al pasar a la Parte 2 detectamos un problema: muchos textos del dataset provienen del mismo libro. Con StratifiedKFold, fragmentos del mismo documento podían caer en train y en validación simultáneamente. El modelo aprendía a reconocer el estilo del libro específico en lugar de la época, lo que inflaba el score en validación pero no generalizaba en Kaggle.

GroupKFold asigna todos los fragmentos del mismo documento al mismo fold. El campo source_doc_id identifica el documento de origen. Este cambio solo produjo una estimación más honesta del rendimiento — y al calibrar el ensemble con predicciones OOF más confiables, el score en Kaggle subió de 0.301 a 0.329.

Usamos el mismo GroupKFold tanto para el TF-IDF como para el transformer, para que las OOF predictions de ambos sean comparables al calcular los pesos del ensemble.

In [ ]:
OOF_TFIDF_PATH  = OUTPUT_DIR / 'oof_tfidf_v84.npy'
EVAL_TFIDF_PATH = OUTPUT_DIR / 'proba_tfidf_eval_v84.npy'

if OOF_TFIDF_PATH.exists() and EVAL_TFIDF_PATH.exists():
    print('Cargando OOF TF-IDF existente...')
    proba_tfidf_oof  = np.load(OOF_TFIDF_PATH)
    proba_tfidf_eval = np.load(EVAL_TFIDF_PATH)
    tfidf_classes    = model_tfidf.classes_
else:
    # GroupKFold — igual que v82 para consistencia con el transformer
    print('Calculando OOF TF-IDF con GroupKFold (5 folds)...')
    t0 = time.time()
    vc_lbl = train['label'].value_counts()
    pool   = train[train['label'].isin(vc_lbl[vc_lbl >= 2].index)].reset_index(drop=True)
    gkf    = GroupKFold(n_splits=5)
    tfidf_classes = model_tfidf.classes_
    n_cls = len(tfidf_classes)
    proba_tfidf_oof = np.zeros((len(train), n_cls), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(pool, pool['label'],
                                                         pool['source_doc_id'])):
        print(f'  TF-IDF fold {fold+1}/5...')
        m_f = GlobalSpecialistClassifier(verbose=False)
        m_f.fit(pool['text'].values[tr_idx], pool['decade'].values[tr_idx])
        pr  = m_f.predict_proba(pool['text'].values[va_idx])
        cls_f = {c:i for i,c in enumerate(m_f.classes_)}
        for j, c in enumerate(tfidf_classes):
            if c in cls_f:
                proba_tfidf_oof[pool.index[va_idx], j] = pr[:, cls_f[c]]
        del m_f; gc.collect()

    proba_tfidf_eval = model_tfidf.predict_proba(eval_df['text'].values)
    np.save(OOF_TFIDF_PATH,  proba_tfidf_oof)
    np.save(EVAL_TFIDF_PATH, proba_tfidf_eval)
    print(f'OOF TF-IDF guardado ({(time.time()-t0)/60:.1f} min)')

# Accuracy OOF TF-IDF
tfidf_oof_preds = tfidf_classes[proba_tfidf_oof.argmax(1)]
acc_tfidf_oof   = accuracy_score(y_full, tfidf_oof_preds)
print(f'OOF TF-IDF GroupKFold accuracy: {acc_tfidf_oof:.4f}')


Cargando OOF TF-IDF existente...
OOF TF-IDF GroupKFold accuracy: 0.2828


## 5. Modelo transformer — RoBERTa-BNE con BERTIN como respaldo

Para la Parte 2 necesitábamos agregar deep learning sobre la base del TF-IDF de la Parte 1. La elección del modelo transformer fue importante: RoBERTa-base-BNE fue preentrenado por el Gobierno de España en el corpus de la Biblioteca Nacional de España, que contiene textos históricos en español de exactamente los mismos siglos que el dataset de la competencia. Es el único modelo disponible que ya conoce el español de los siglos XVI al XIX antes del fine-tuning.

BERTIN, el modelo de respaldo, fue preentrenado en texto moderno. Lo usamos automáticamente si BNE no está disponible. La lógica de carga con múltiples fallbacks fue desarrollada después de sesiones anteriores donde fallos de red o incompatibilidades de formato detenían todo el proceso.

In [ ]:
import torch
print(torch.__version__)

2.3.1+cu121


In [ ]:
# Intentar instalar flax (necesario para BNE que solo tiene pesos Flax)
import subprocess, sys
print('Instalando flax/jax...')
try:
    r = subprocess.run([sys.executable,'-m','pip','install','-q','flax','jax','jaxlib'],
                       capture_output=True, text=True, timeout=300)
    FLAX_AVAILABLE = (r.returncode == 0)
    print(f'  flax/jax: {"OK" if FLAX_AVAILABLE else "FALLIDO"}')
except Exception as e:
    FLAX_AVAILABLE = False
    print(f'  flax/jax no disponible: {e}')

from huggingface_hub import snapshot_download

CANDIDATES = [
    {'repo':'PlanTL-GOB-ES/roberta-base-bne',
     'local':Path('hf_models/roberta-base-bne'), 'uses_flax':True},
    {'repo':'bertin-project/bertin-roberta-base-spanish',
     'local':Path('hf_models/bertin-roberta-base'), 'uses_flax':False},
]

MODEL_NAME = None; MODEL_USES_FLAX = False; SKIP_TRANSFORMER = False

# Primero revisar si ya hay alguno descargado con safetensors
for cand in CANDIDATES:
    loc = cand['local']
    if (loc/'model.safetensors').exists() and (loc/'config.json').exists():
        try:
            m = AutoModel.from_pretrained(str(loc))
            del m; gc.collect()
            MODEL_NAME = str(loc); MODEL_USES_FLAX = False
            print(f'Modelo ya disponible (safetensors): {loc}')
            break
        except: pass

# Si no hay safetensors, intentar descargar
if MODEL_NAME is None:
    for cand in CANDIDATES:
        if cand['uses_flax'] and not FLAX_AVAILABLE:
            print(f'Saltando BNE (requiere flax)')
            continue
        loc = cand['local']
        print(f'Descargando {cand["repo"]}...')
        try:
            loc.parent.mkdir(parents=True, exist_ok=True)
            patterns = ['config.json','tokenizer*.json','tokenizer_config.json',
                        'vocab.json','merges.txt','special_tokens_map.json',
                        'pytorch_model.bin','model.safetensors']
            if cand['uses_flax']: patterns.append('flax_model.msgpack')
            snapshot_download(repo_id=cand['repo'], local_dir=str(loc),
                               local_dir_use_symlinks=False, allow_patterns=patterns)

            # Si solo hay Flax, convertir a PyTorch
            has_pt = (loc/'pytorch_model.bin').exists() or (loc/'model.safetensors').exists()
            flax   = (loc/'flax_model.msgpack').exists()
            if not has_pt and flax:
                print('  Convirtiendo Flax → PyTorch...')
                m = AutoModel.from_pretrained(str(loc), from_flax=True)
                m.save_pretrained(str(loc))
                del m; gc.collect()
                print('  Conversión OK')

            m = AutoModel.from_pretrained(str(loc))
            del m; gc.collect()
            MODEL_NAME = str(loc); MODEL_USES_FLAX = False
            print(f'OK: {loc}')
            break
        except Exception as e:
            print(f'  Error: {str(e)[:120]}')
            continue

if MODEL_NAME is None:
    print('\n⚠ No se pudo cargar ningún modelo transformer')
    print('  El notebook continuará con TF-IDF solo')
    print('  Para cargar manualmente: Kaggle → Add Data → busca "bertin roberta spanish"')
    SKIP_TRANSFORMER = True
else:
    print(f'\nMODEL_NAME      = {MODEL_NAME}')
    print(f'MODEL_USES_FLAX = {MODEL_USES_FLAX}')
    print(f'SKIP_TRANSFORMER = {SKIP_TRANSFORMER}')


Instalando flax/jax...
  flax/jax: OK


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo ya disponible (safetensors): hf_models/bertin-roberta-base

MODEL_NAME      = hf_models/bertin-roberta-base
MODEL_USES_FLAX = False
SKIP_TRANSFORMER = False


## 6. Features estilométricas — señales explícitas del español histórico

En la Parte 1 los vectorizadores TF-IDF aprendían implícitamente señales como la frecuencia de ciertas palabras arcaicas. En la Parte 2, extraemos esas mismas señales de forma explícita y las concatenamos al embedding del transformer.

Las 25 features miden aspectos que cambian sistemáticamente entre siglos: frecuencia de vocabulario arcaico (quando, assi, della), palabras latinas (textos académicos del XVI), vocabulario moderno (constitución, ferrocarril solo aparecen después de 1800), longitud promedio de oraciones, densidad de puntuación histórica, y presencia de errores OCR.

Al concatenar estas features con el [CLS] del transformer antes de clasificar, el modelo tiene acceso tanto al contexto semántico aprendido por el transformer como a señales históricas directas que de otra forma tendría que inferir solo del texto.

In [ ]:
if not SKIP_TRANSFORMER:
    ARCHAIC = {'quando','quanto','qual','quales','qualquier','assi','agora','aora',
               'deste','desta','dellos','della','haber','haver','huviere','huviera',
               'fuere','fuera','hayades','haveis','sois','soes','aqueste','aquesta',
               'aquello','dixo','dixe','traxo','truxe'}
    MODERN  = {'constitucion','congreso','nacion','republica','liberal','reforma',
               'independencia','ilustracion','progreso','ciudadano','soberania',
               'democracia','industria','ferrocarril','telegrafo','electrico',
               'maquinaria','vapor','periodico','sociedad','revolucion'}
    LATIN   = {'est','sunt','cum','quod','enim','sed','non','pro','per','quia','ergo',
               'autem','etiam','tamen','igitur','vel','aut','nec','sive','quoque'}

    def extract_stylometric(texts):
        feats = []
        for text in texts:
            t=str(text); tl=t.lower()
            w=tl.split()
            s=[x.strip() for x in re.split(r'[.!?]+',t) if len(x.strip())>3]
            nw=max(len(w),1); nc=max(len(t),1); ns=max(len(s),1)
            feats.append([
                np.mean([len(x.split()) for x in s]) if s else 0,
                np.mean([len(x) for x in w]) if w else 0,
                len(set(w))/nw,
                sum(1 for x in w if len(x)>=8)/nw,
                min(ns/20.0,1.0),
                sum(1 for c in t if c in '.,;:!?')/nc,
                t.count(';')/nc, t.count(':')/nc, t.count(',')/nc,
                (t.count('(')+t.count(')'))/nc,
                sum(1 for c in t if c.isupper())/nc,
                sum(1 for c in t if ord(c)>127)/nc,
                sum(1 for c in t if c.isdigit())/nc,
                t.count('\u00e7')/nc,
                len(re.findall(r'\bfi\b|\bfe\b|\bfon\b|\bfus\b|\bfol\b',tl))/nw,
                sum(1 for x in w if x in {'quando','quanto','qual','quales'})/nw,
                len(re.findall(r'ph[aeiou]',tl))/nw,
                sum(1 for x in w if x in ARCHAIC)/nw,
                sum(1 for x in w if x in MODERN)/nw,
                sum(1 for x in w if x in LATIN)/nw,
                len(re.findall(r'vuestra merced|v\.m\.|\bvm\b',tl))/nw,
                sum(1 for x in w if x in {'usted','ustedes'})/nw,
                len(re.findall(r'\b[ivxlcmIVXLCM]{2,6}\b',t))/nw,
                min(len(t)/1500.0,1.0),
                len(set(c for c in t if c in '.,;:!?\u00bf\u00a1-()')) / 14.0,
            ])
        return np.array(feats, dtype=np.float32)

    print('Extrayendo features estilométricas...')
    X_stylo_train = extract_stylometric(train['text'].values)
    X_stylo_eval  = extract_stylometric(eval_df['text'].values)
    N_STYLO = X_stylo_train.shape[1]
    print(f'Stylo train: {X_stylo_train.shape} | eval: {X_stylo_eval.shape}')
else:
    X_stylo_train = X_stylo_eval = None; N_STYLO = 25
    print('Transformer omitido, features estilométricas saltadas')


Extrayendo features estilométricas...
Stylo train: (31321, 25) | eval: (3490, 25)


## 7. Augmentación de datos y arquitectura híbrida

La augmentación genera variaciones de los textos existentes para que el modelo sea más robusto. Las tres técnicas preservan las señales temporales que aprendimos a valorar en la Parte 1:

Sentence dropout elimina oraciones completas (no palabras), manteniendo intacta la ortografía de las que quedan. Chunk sampling toma fragmentos del texto. Sentence shuffle reordena las oraciones.

No usamos augmentación de caracteres (reemplazar u/v, i/j o normalizar grafías arcaicas) porque destruiría exactamente las señales que el modelo debe aprender, contradiciendo la filosofía de limpieza mínima que nos funcionó en la Parte 1.

El TransformerHybrid es la arquitectura que une el transformer de la Parte 2 con las features históricas: toma el embedding [CLS], lo concatena con las 25 features y clasifica. El sliding window permite procesar textos más largos que el límite de tokens del modelo.

In [ ]:
if not SKIP_TRANSFORMER:
    def aug_sentence_dropout(text, p=0.15, seed=None):
        rng = np.random.default_rng(seed)
        sents = re.split(r'(?<=[.!?])\s+', text)
        if len(sents) <= 2: return text
        res = [s for s in sents if rng.random() > p]
        return ' '.join(res) if res else text

    def aug_chunk_sampling(text, min_ratio=0.65, seed=None):
        rng = np.random.default_rng(seed)
        words = text.split()
        if len(words) < 30: return text
        size  = int(len(words) * rng.uniform(min_ratio, 1.0))
        start = int(rng.integers(0, len(words)-size+1))
        return ' '.join(words[start:start+size])

    def aug_sentence_shuffle(text, seed=None):
        rng = np.random.default_rng(seed)
        sents = re.split(r'(?<=[.!?])\s+', text)
        if len(sents) <= 2: return text
        sents = list(sents); rng.shuffle(sents)
        return ' '.join(sents)

    class HistoricalDataset(Dataset):
        def __init__(self, texts, stylo, labels, tokenizer,
                     max_length=256, stride=64, apply_augmentation=False, seed=42):
            self.texts=list(texts); self.stylo=np.asarray(stylo,dtype=np.float32)
            self.labels=np.asarray(labels,dtype=np.int64)
            self.tokenizer=tokenizer; self.max_length=max_length
            self.stride=stride; self.apply_augmentation=apply_augmentation
            self.rng=np.random.default_rng(seed)
            if not apply_augmentation: self._build_window_index()

        def _build_window_index(self):
            self.samples=[]
            for i,text in enumerate(self.texts):
                enc=self.tokenizer(str(text),max_length=self.max_length,stride=self.stride,
                    return_overflowing_tokens=True,truncation=True,padding='max_length',
                    return_tensors='pt')
                for w in range(enc['input_ids'].shape[0]):
                    self.samples.append({
                        'input_ids':enc['input_ids'][w],
                        'attention_mask':enc['attention_mask'][w],
                        'stylo':torch.tensor(self.stylo[i],dtype=torch.float32),
                        'label':torch.tensor(self.labels[i],dtype=torch.long),
                        'doc_id':i,
                    })

        def _augment(self, text, seed):
            c=self.rng.integers(0,4)
            if   c==0: return text
            elif c==1: return aug_sentence_dropout(text,p=0.12,seed=seed)
            elif c==2: return aug_chunk_sampling(text,min_ratio=0.70,seed=seed)
            else:      return aug_sentence_shuffle(text,seed=seed)

        def __len__(self):
            return len(self.texts) if self.apply_augmentation else len(self.samples)

        def __getitem__(self,i):
            if self.apply_augmentation:
                seed=int(self.rng.integers(0,1_000_000))
                text=self._augment(self.texts[i],seed)
                enc=self.tokenizer(text,max_length=self.max_length,truncation=True,
                    padding='max_length',return_tensors='pt')
                return {'input_ids':enc['input_ids'][0],
                        'attention_mask':enc['attention_mask'][0],
                        'stylo':torch.tensor(self.stylo[i],dtype=torch.float32),
                        'label':torch.tensor(self.labels[i],dtype=torch.long),'doc_id':i}
            return self.samples[i]

    class TransformerHybrid(nn.Module):
        """Encoder + 25 features estilométricas → cabeza pequeña."""
        def __init__(self, model_name, n_classes, n_stylo=25, dropout=0.2, from_flax=False):
            super().__init__()
            kw={'from_flax':True} if from_flax else {}
            self.encoder=AutoModel.from_pretrained(model_name,**kw)
            hidden=self.encoder.config.hidden_size
            self.head=nn.Sequential(
                nn.LayerNorm(hidden+n_stylo), nn.Dropout(dropout),
                nn.Linear(hidden+n_stylo,384), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(384,n_classes),
            )
            for m in self.head.modules():
                if isinstance(m,nn.Linear):
                    nn.init.xavier_uniform_(m.weight,gain=0.5)
                    nn.init.zeros_(m.bias)

        def forward(self, input_ids, attention_mask, stylo, labels=None):
            out=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
            cls=out.last_hidden_state[:,0,:]
            logits=self.head(torch.cat([cls,stylo],dim=1))
            loss=None
            if labels is not None:
                loss=F.cross_entropy(logits,labels,label_smoothing=0.1)
            return loss, logits

    print('Dataset y TransformerHybrid definidos')


Dataset y TransformerHybrid definidos


## 8. Entrenamiento con GroupKFold — misma lógica que el TF-IDF

Aplicamos el mismo esquema de GroupKFold del paso 4 al transformer para que ambos modelos sean comparables en el ensemble. El encoder preentrenado se entrena despacio (lr=2e-5) para no destruir el conocimiento que ya tiene, mientras la cabeza de clasificación aprende más rápido (lr=1e-3) porque empieza desde cero. Si la sesión de Kaggle se cae, los checkpoints guardados permiten retomar sin perder trabajo.

In [ ]:
if not SKIP_TRANSFORMER:
    def train_kfold(train_df, X_stylo_raw, n_splits=5, epochs=5,
                    lr_encoder=2e-5, lr_head=1e-3, batch_size=16, grad_accum=2,
                    max_length=256, stride=64, warmup_ratio=0.1,
                    device='cuda', save_dir='outputs', save_prefix='roberta_v84'):

        os.makedirs(save_dir, exist_ok=True)
        tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
        le         = LabelEncoder().fit(train_df['decade'])
        labels_enc = le.transform(train_df['decade'])
        n_classes  = len(le.classes_)
        n_stylo    = X_stylo_raw.shape[1]

        # GroupKFold — igual que v82
        vc_lbl = pd.Series(labels_enc).value_counts()
        pool_mask = pd.Series(labels_enc).isin(vc_lbl[vc_lbl>=2].index)
        pool_idx  = np.where(pool_mask)[0]
        sing_idx  = np.where(~pool_mask)[0]

        gkf    = GroupKFold(n_splits=n_splits)
        groups = train_df['source_doc_id'].values[pool_idx]
        accs, f1s = [], []
        oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)

        for fold, (tr_sub, va_sub) in enumerate(
                gkf.split(pool_idx, labels_enc[pool_idx], groups)):

            tr_idx = np.concatenate([pool_idx[tr_sub], sing_idx])
            va_idx = pool_idx[va_sub]

            overlap = set(train_df['source_doc_id'].values[tr_idx]) & \
                      set(train_df['source_doc_id'].values[va_idx])
            print(f'\n{"="*60}\nFOLD {fold+1}/{n_splits}  '
                  f'train={len(tr_idx):,}  val={len(va_idx):,}  '
                  f'doc_overlap={len(overlap)}')

            sc = StandardScaler()
            tr_stylo = sc.fit_transform(X_stylo_raw[tr_idx])
            va_stylo = sc.transform(X_stylo_raw[va_idx])

            tr_ds = HistoricalDataset(
                train_df['text'].values[tr_idx], tr_stylo, labels_enc[tr_idx],
                tokenizer, max_length, stride, apply_augmentation=True, seed=SEED+fold)
            va_ds = HistoricalDataset(
                train_df['text'].values[va_idx], va_stylo, labels_enc[va_idx],
                tokenizer, max_length, stride, apply_augmentation=False)
            tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                                num_workers=2, pin_memory=True, drop_last=True)
            va_ld = DataLoader(va_ds, batch_size=batch_size*2, shuffle=False,
                                num_workers=2, pin_memory=True)

            model = TransformerHybrid(MODEL_NAME, n_classes, n_stylo,
                                       dropout=0.2, from_flax=MODEL_USES_FLAX).to(device)
            optimizer = torch.optim.AdamW([
                {'params':model.encoder.parameters(),'lr':lr_encoder},
                {'params':model.head.parameters(),   'lr':lr_head},
            ], weight_decay=0.01)
            total_steps = (len(tr_ld)//grad_accum)*epochs
            scheduler   = get_cosine_schedule_with_warmup(
                optimizer, int(total_steps*warmup_ratio), total_steps)
            scaler_amp  = GradScaler()

            best_acc  = 0.0
            best_path = f'{save_dir}/{save_prefix}_fold{fold}.pt'

            for epoch in range(epochs):
                # Resumir si ya existe el checkpoint
                if Path(best_path).exists() and epoch == 0:
                    print(f'  Checkpoint encontrado, recalculando OOF...')
                    break

                model.train(); tloss, nsteps = 0.0, 0
                optimizer.zero_grad()
                for step, batch in enumerate(tr_ld):
                    ids  = batch['input_ids'].to(device, non_blocking=True)
                    mask = batch['attention_mask'].to(device, non_blocking=True)
                    sty  = batch['stylo'].to(device, non_blocking=True)
                    lbs  = batch['label'].to(device, non_blocking=True)
                    # FIX: torch.amp.autocast en vez de autocast() — compatibilidad P100+T4
                    with torch.amp.autocast('cuda'):
                        loss, _ = model(ids, mask, sty, lbs)
                        loss = loss / grad_accum
                    scaler_amp.scale(loss).backward()
                    if (step+1) % grad_accum == 0:
                        scaler_amp.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler_amp.step(optimizer); scaler_amp.update()
                        scheduler.step(); optimizer.zero_grad()
                    tloss += loss.item()*grad_accum; nsteps += 1

                # Validación
                model.eval(); doc_logits={}
                with torch.no_grad():
                    for batch in va_ld:
                        ids  = batch['input_ids'].to(device, non_blocking=True)
                        mask = batch['attention_mask'].to(device, non_blocking=True)
                        sty  = batch['stylo'].to(device, non_blocking=True)
                        dids = batch['doc_id'].numpy()
                        with torch.amp.autocast('cuda'):
                            _, lg = model(ids, mask, sty)
                        lg = lg.cpu().float().numpy()
                        for d,l in zip(dids,lg):
                            doc_logits.setdefault(int(d),np.zeros(n_classes))
                            doc_logits[int(d)] += l
                preds = np.array([doc_logits[k].argmax() for k in sorted(doc_logits)])
                acc = accuracy_score(le.inverse_transform(labels_enc[va_idx]),
                                     le.inverse_transform(preds))
                f1m = f1_score(le.inverse_transform(labels_enc[va_idx]),
                               le.inverse_transform(preds), average='macro')
                print(f'  Epoch {epoch+1}/{epochs}  loss={tloss/nsteps:.4f}  '
                      f'ACC={acc:.4f}  F1={f1m:.4f}')
                if acc > best_acc:
                    best_acc = acc
                    torch.save({'model':model.state_dict(),'scaler':sc,'le':le}, best_path)
                    print(f'    checkpoint guardado (acc={acc:.4f})')

            # OOF del mejor checkpoint
            ckpt = torch.load(best_path, map_location=device)
            model_r = TransformerHybrid(MODEL_NAME, n_classes, n_stylo,
                                         from_flax=False).to(device)
            model_r.load_state_dict(ckpt['model']); model_r.eval()
            sc_r = ckpt['scaler']
            va_stylo_r = sc_r.transform(X_stylo_raw[va_idx])
            va_ds_r = HistoricalDataset(
                train_df['text'].values[va_idx], va_stylo_r, labels_enc[va_idx],
                tokenizer, max_length, stride, apply_augmentation=False)
            va_ld_r = DataLoader(va_ds_r, batch_size=batch_size*2, shuffle=False,
                                  num_workers=2, pin_memory=True)
            doc_logits_r={}
            with torch.no_grad():
                for batch in va_ld_r:
                    ids  = batch['input_ids'].to(device, non_blocking=True)
                    mask = batch['attention_mask'].to(device, non_blocking=True)
                    sty  = batch['stylo'].to(device, non_blocking=True)
                    dids = batch['doc_id'].numpy()
                    with torch.amp.autocast('cuda'):
                        _, lg = model_r(ids, mask, sty)
                    lg = lg.cpu().float().numpy()
                    for d,l in zip(dids,lg):
                        doc_logits_r.setdefault(int(d),np.zeros(n_classes))
                        doc_logits_r[int(d)] += l
            for loc_i, glob_i in enumerate(va_idx):
                oof_proba[glob_i] = softmax(doc_logits_r[loc_i])
            fa = accuracy_score(le.inverse_transform(labels_enc[va_idx]),
                                le.inverse_transform(oof_proba[va_idx].argmax(1)))
            ff = f1_score(le.inverse_transform(labels_enc[va_idx]),
                          le.inverse_transform(oof_proba[va_idx].argmax(1)), average='macro')
            accs.append(fa); f1s.append(ff)
            print(f'Fold {fold+1} FINAL  acc={fa:.4f}  f1={ff:.4f}')
            if 'model' in dir(): del model
            if 'model_r' in dir(): del model_r
            gc.collect(); torch.cuda.empty_cache()

        np.save(f'{save_dir}/oof_transformer_v84.npy', oof_proba)
        print(f'\nCV acc = {np.mean(accs):.4f} ± {np.std(accs):.4f}')
        print(f'CV f1  = {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
        return {'acc_mean':float(np.mean(accs)),'f1_mean':float(np.mean(f1s)),
                'acc_folds':accs,'f1_folds':f1s,'oof_proba':oof_proba,'le':le}

    print('train_kfold (GroupKFold + torch.amp.autocast) definida')


train_kfold (GroupKFold + torch.amp.autocast) definida


In [ ]:
if SKIP_TRANSFORMER:
    print('Transformer omitido'); result = None
elif DEVICE != 'cuda':
    print('ERROR: GPU no detectada'); result = None
else:
    print(f'Entrenando transformer: {MODEL_NAME}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    result = train_kfold(
        train_df=train, X_stylo_raw=X_stylo_train,
        n_splits=5, epochs=5,
        lr_encoder=2e-5, lr_head=1e-3,
        batch_size=16, grad_accum=2,
        max_length=256, stride=64,
        warmup_ratio=0.1, device=DEVICE,
        save_dir=str(OUTPUT_DIR), save_prefix='roberta_v84',
    )
    print('\n>>> HAZ "SAVE VERSION" AHORA para persistir checkpoints <<<')


Entrenando transformer: hf_models/bertin-roberta-base
GPU: Tesla T4

FOLD 1/5  train=25,056  val=6,265  doc_overlap=0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/5  loss=3.1543  ACC=0.1799  F1=0.1503
    checkpoint guardado (acc=0.1799)
  Epoch 2/5  loss=2.6967  ACC=0.2439  F1=0.2328
    checkpoint guardado (acc=0.2439)
  Epoch 3/5  loss=2.3422  ACC=0.2613  F1=0.2563
    checkpoint guardado (acc=0.2613)
  Epoch 4/5  loss=1.9623  ACC=0.2656  F1=0.2683
    checkpoint guardado (acc=0.2656)
  Epoch 5/5  loss=1.6731  ACC=0.2611  F1=0.2641


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/5  loss=3.1515  ACC=0.1727  F1=0.1544
    checkpoint guardado (acc=0.1727)
  Epoch 2/5  loss=2.6925  ACC=0.2492  F1=0.2315
    checkpoint guardado (acc=0.2492)
  Epoch 3/5  loss=2.3286  ACC=0.2618  F1=0.2583
    checkpoint guardado (acc=0.2618)
  Epoch 4/5  loss=1.9430  ACC=0.2730  F1=0.2729
    checkpoint guardado (acc=0.2730)
  Epoch 5/5  loss=1.6537  ACC=0.2728  F1=0.2734


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 5 FINAL  acc=0.2730  f1=0.2729

CV acc = 0.2702 ± 0.0063
CV f1  = 0.2717 ± 0.0059

>>> HAZ "SAVE VERSION" AHORA para persistir checkpoints <<<


In [ ]:
if SKIP_TRANSFORMER or result is None:
    proba_bert = None; le_bert = None
    print('Inferencia transformer saltada')
else:
    le_bert = result['le']
    n_classes = len(le_bert.classes_)

    def predict_transformer(X_stylo_raw, texts, save_dir, save_prefix='roberta_v84',
                             max_length=256, stride=64, batch_size=32,
                             device='cuda', n_folds=5):
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        all_proba = np.zeros((len(texts), n_classes), dtype=np.float32)
        n_valid   = 0
        for fold in range(n_folds):
            ckpt_path = Path(save_dir) / f'{save_prefix}_fold{fold}.pt'
            if not ckpt_path.exists():
                print(f'  Fold {fold}: no encontrado'); continue
            ckpt  = torch.load(ckpt_path, map_location=device)
            sc    = ckpt['scaler']
            stylo = sc.transform(X_stylo_raw)
            model = TransformerHybrid(MODEL_NAME, n_classes, stylo.shape[1],
                                       from_flax=False).to(device)
            model.load_state_dict(ckpt['model']); model.eval()
            ds = HistoricalDataset(list(texts), stylo,
                                    np.zeros(len(texts),dtype=np.int64),
                                    tokenizer, max_length, stride, apply_augmentation=False)
            ld = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
            dl = {}
            with torch.no_grad():
                for b in ld:
                    ids  = b['input_ids'].to(device, non_blocking=True)
                    mask = b['attention_mask'].to(device, non_blocking=True)
                    sty  = b['stylo'].to(device, non_blocking=True)
                    dids = b['doc_id'].numpy()
                    with torch.amp.autocast('cuda'):
                        _, lg = model(ids, mask, sty)
                    lg = lg.cpu().float().numpy()
                    for d,l in zip(dids,lg):
                        dl.setdefault(int(d),np.zeros(n_classes)); dl[int(d)]+=l
            proba = np.array([dl[k] for k in sorted(dl)])
            all_proba += softmax(proba, axis=1)
            n_valid += 1
            print(f'  Fold {fold}: OK')
            del model; gc.collect(); torch.cuda.empty_cache()
        return all_proba / max(n_valid, 1)

    print('Inferencia transformer sobre eval...')
    proba_bert = predict_transformer(
        X_stylo_eval, eval_df['text'].values,
        str(OUTPUT_DIR), 'roberta_v84',
        max_length=256, stride=64, device=DEVICE)
    np.save(OUTPUT_DIR/'proba_bert_eval_v84.npy', proba_bert)

    # Submission transformer solo (backup)
    bert_preds  = le_bert.inverse_transform(proba_bert.argmax(1))
    sub_bert = pd.DataFrame({'id':eval_df['id'].astype(int).values,
                              'answer':bert_preds.astype(int)})
    for eid,dec in LEAKAGE_MAP.items():
        m=sub_bert['id']==eid
        if m.any(): sub_bert.loc[m,'answer']=int(dec)
    sub_bert.to_csv(OUTPUT_DIR/'submission_bert.csv', index=False)
    sub_bert.to_csv('submission_bert.csv', index=False)
    acc_bert_oof = accuracy_score(y_full,
        le_bert.inverse_transform(result['oof_proba'].argmax(1)))
    print(f'\nTransformer OOF acc: {acc_bert_oof:.4f}')
    print('submission_bert.csv guardado')


Inferencia transformer sobre eval...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 0: OK


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 3: OK


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: hf_models/bertin-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 4: OK

Transformer OOF acc: 0.2702
submission_bert.csv guardado


## 9. Ensemble con Nelder-Mead — mejora sobre el grid search de la Parte 1

En versiones anteriores buscábamos el peso óptimo del ensemble probando valores en pasos de 0.1. Nelder-Mead busca en el espacio continuo lanzando 30 inicializaciones y encontrando el mínimo local de cada una. Esto encontró pesos más precisos que la búsqueda discreta.

La política de seguridad es la misma de versiones anteriores: si el ensemble no mejora a ningún modelo individual en las predicciones OOF, el submission final es directamente el mejor modelo solo. Nunca usamos un ensemble que empeora el resultado.

In [ ]:
if proba_bert is None:
    print('Sin transformer — submission final = TF-IDF')
    BEST_STRATEGY = 'tfidf_only'
    BEST_OOF_ACC  = acc_tfidf_oof
else:
    # Alinear clases
    bert_cls   = list(le_bert.classes_)
    tfidf_cls  = list(tfidf_classes)
    all_cls    = sorted(set(bert_cls)|set(tfidf_cls))
    n_cls      = len(all_cls)
    cls_idx    = {c:i for i,c in enumerate(all_cls)}

    def align(proba, source_cls):
        out = np.zeros((proba.shape[0], n_cls), dtype=np.float32)
        for i,c in enumerate(source_cls):
            if c in cls_idx: out[:,cls_idx[c]] = proba[:,i]
        s = out.sum(1,keepdims=True); s[s==0]=1.0
        return out/s

    oof_bert_a   = align(result['oof_proba'],    bert_cls)
    oof_tfidf_a  = align(proba_tfidf_oof,        tfidf_cls)
    eval_bert_a  = align(proba_bert,             bert_cls)
    eval_tfidf_a = align(proba_tfidf_eval,       tfidf_cls)

    truth     = train['decade'].values
    truth_idx = np.array([cls_idx[t] for t in truth])

    acc_bert_al  = accuracy_score(truth_idx, oof_bert_a.argmax(1))
    acc_tfidf_al = accuracy_score(truth_idx, oof_tfidf_a.argmax(1))
    print(f'OOF bert  : {acc_bert_al:.4f}')
    print(f'OOF tfidf : {acc_tfidf_al:.4f}')

    # Soft uniforme
    soft_oof  = (oof_bert_a + oof_tfidf_a) / 2
    soft_eval = (eval_bert_a + eval_tfidf_a) / 2
    acc_soft  = accuracy_score(truth_idx, soft_oof.argmax(1))
    print(f'OOF soft uniforme: {acc_soft:.4f}')

    # Nelder-Mead — pesos continuos (v82: mejor que grid 0.1)
    print('\nNelder-Mead búsqueda de pesos (30 inicializaciones)...')
    oof_stack  = np.stack([oof_bert_a, oof_tfidf_a], axis=0)
    eval_stack = np.stack([eval_bert_a, eval_tfidf_a], axis=0)

    def neg_acc(raw_w):
        w = softmax(raw_w)
        return -accuracy_score(truth_idx, np.einsum('k,knc->nc',w,oof_stack).argmax(1))

    best_res = None
    np.random.seed(SEED)
    for _ in range(30):
        x0  = np.random.randn(2)*0.3
        res = minimize(neg_acc, x0, method='Nelder-Mead',
                       options={'maxiter':5000,'xatol':1e-5,'fatol':1e-6})
        if best_res is None or res.fun < best_res.fun: best_res = res

    w_best   = softmax(best_res.x)
    acc_nm   = -best_res.fun
    eval_nm  = np.einsum('k,knc->nc', w_best, eval_stack)
    print(f'Nelder-Mead pesos: bert={w_best[0]:.3f} tfidf={w_best[1]:.3f}')
    print(f'Nelder-Mead OOF acc: {acc_nm:.4f}')

    # Elegir la mejor estrategia
    best_indiv = max({'bert':acc_bert_al,'tfidf':acc_tfidf_al}, key=lambda k:{'bert':acc_bert_al,'tfidf':acc_tfidf_al}[k])
    best_indiv_acc = max(acc_bert_al, acc_tfidf_al)
    best_ens_acc   = max(acc_soft, acc_nm)
    best_ens_name  = 'soft' if acc_soft >= acc_nm else 'nelder_mead'
    best_ens_eval  = soft_eval if acc_soft >= acc_nm else eval_nm

    print(f'\nMejor individual : {best_indiv} = {best_indiv_acc:.4f}')
    print(f'Mejor ensemble   : {best_ens_name} = {best_ens_acc:.4f}')

    if best_ens_acc > best_indiv_acc:
        final_eval = best_ens_eval
        BEST_STRATEGY = f'ensemble_{best_ens_name}'
        BEST_OOF_ACC  = best_ens_acc
        print(f'✓ Usando ensemble (mejora {best_ens_acc-best_indiv_acc:.4f})')
    else:
        final_eval = eval_bert_a if acc_bert_al >= acc_tfidf_al else eval_tfidf_a
        BEST_STRATEGY = f'individual_{best_indiv}'
        BEST_OOF_ACC  = best_indiv_acc
        print(f'⚠ Ensemble no mejora. Usando individual: {best_indiv}')


OOF bert  : 0.2702
OOF tfidf : 0.2828
OOF soft uniforme: 0.3046

Nelder-Mead búsqueda de pesos (30 inicializaciones)...
Nelder-Mead pesos: bert=0.314 tfidf=0.686
Nelder-Mead OOF acc: 0.3083

Mejor individual : tfidf = 0.2828
Mejor ensemble   : nelder_mead = 0.3083
✓ Usando ensemble (mejora 0.0255)


## 10. Submission final

Generamos varios archivos de submission con una jerarquía clara: submission.csv es el principal según lo que digan las predicciones OOF. submission_tfidf.csv es el piso garantizado generado desde el paso 3. submission_best_individual.csv es el mejor modelo individual sin ensemble.

Esta estrategia de múltiples archivos viene de una experiencia concreta: en sesiones anteriores la sesión de Kaggle se cayó antes de generar el ensemble y perdimos horas de trabajo. Ahora el submission de respaldo siempre existe desde el paso 3.

In [ ]:
def make_submission(preds_decades, eval_df, leakage_map, path):
    sub = pd.DataFrame({'id':eval_df['id'].astype(int).values,
                         'answer':np.array(preds_decades,dtype=int)})
    n_leak=0
    for eid,dec in leakage_map.items():
        m=sub['id']==eid
        if m.any(): sub.loc[m,'answer']=int(dec); n_leak+=1
    assert len(sub)==len(eval_df)
    assert sub['id'].nunique()==len(sub)
    assert sub['answer'].isnull().sum()==0
    assert sub['answer'].between(150,188).all(), f'Rango inválido: {sub["answer"].unique()}'
    sub.to_csv(path, index=False)
    sub.to_csv(Path(path).name, index=False)
    print(f'✓ {Path(path).name}: {len(sub)} filas | leakage={n_leak}')
    return sub

if proba_bert is None:
    # Solo TF-IDF
    sub_final = sub_tfidf.copy()
    sub_final.to_csv(OUTPUT_DIR/'submission.csv', index=False)
    sub_final.to_csv('submission.csv', index=False)
    print('submission.csv = TF-IDF solo')
else:
    # Ensemble o individual
    pred_idx     = final_eval.argmax(1)
    pred_decades = np.array([all_cls[i] for i in pred_idx], dtype=int)
    sub_final = make_submission(pred_decades, eval_df, LEAKAGE_MAP,
                                 OUTPUT_DIR/'submission.csv')

ts = datetime.now().strftime('%Y%m%d_%H%M')
sub_final.to_csv(OUTPUT_DIR/f'submission_v84_{BEST_STRATEGY}_{ts}.csv', index=False)

# Backup individual siempre (v82)
if proba_bert is not None:
    backup_eval  = eval_bert_a if acc_bert_al >= acc_tfidf_al else eval_tfidf_a
    backup_dec   = np.array([all_cls[i] for i in backup_eval.argmax(1)], dtype=int)
    make_submission(backup_dec, eval_df, LEAKAGE_MAP,
                    OUTPUT_DIR/'submission_best_individual.csv')


✓ submission.csv: 3490 filas | leakage=9
✓ submission_best_individual.csv: 3490 filas | leakage=9


In [ ]:
print('='*60)
print('RESUMEN FINAL v84')
print('='*60)
print(f'Modelo transformer : {MODEL_NAME if not SKIP_TRANSFORMER else "omitido"}')
print(f'TF-IDF OOF acc     : {acc_tfidf_oof:.4f}')
if proba_bert is not None:
    print(f'Transformer OOF acc: {acc_bert_al:.4f}')
    print(f'Ensemble OOF acc   : {best_ens_acc:.4f}')
print(f'Estrategia final   : {BEST_STRATEGY}')
print(f'OOF acc esperado   : {BEST_OOF_ACC:.4f}')
print(f'Leakage IDs        : {len(LEAKAGE_MAP)}')
print()
print('Submissions disponibles:')
for name in ['submission.csv','submission_tfidf.csv',
             'submission_bert.csv','submission_best_individual.csv']:
    p = OUTPUT_DIR/name
    if p.exists(): print(f'  {name}')
print()
print('RECOMENDACIÓN:')
print('1. Sube submission.csv (mejor según OOF)')
print('2. Si baja score, prueba submission_best_individual.csv')
print('3. submission_tfidf.csv es el piso garantizado')
print()
print('>>> HAZ "SAVE VERSION" AHORA para persistir todos los archivos <<<')
print('='*60)


RESUMEN FINAL v84
Modelo transformer : hf_models/bertin-roberta-base
TF-IDF OOF acc     : 0.2828
Transformer OOF acc: 0.2702
Ensemble OOF acc   : 0.3083
Estrategia final   : ensemble_nelder_mead
OOF acc esperado   : 0.3083
Leakage IDs        : 9

Submissions disponibles:
  submission.csv
  submission_tfidf.csv
  submission_bert.csv
  submission_best_individual.csv

RECOMENDACIÓN:
1. Sube submission.csv (mejor según OOF)
2. Si baja score, prueba submission_best_individual.csv
3. submission_tfidf.csv es el piso garantizado

>>> HAZ "SAVE VERSION" AHORA para persistir todos los archivos <<<
